# 재무 Application을 위한 Claude Skills

Claude의 Excel, PowerPoint, PDF skill을 사용해 실제 재무 dashboard, portfolio analytics, 자동 보고 workflow를 구축합니다.

> **💡 실제 영향:** 이들은 **[Claude의 파일 생성](https://www.anthropic.com/news/create-files)**을 뒷받침하는 동일한 Skills로, interface에서 직접 전문적인 재무 문서를 만들 수 있게 합니다.

**배울 내용:**
- formula와 chart가 있는 종합적인 재무 model을 Excel에서 생성
- 재무 data에서 경영진 presentation 생성
- risk metric이 있는 portfolio 분석 도구 구축
- 여러 형식의 보고 pipeline 자동화

## 목차

1. [설정 및 Data 로딩](#setup)
2. [사용 사례 1: 재무 Dashboard 생성](#financial-dashboard)
   - [Excel 재무 Model](#excel-model)
   - [경영진 PowerPoint](#executive-ppt)
   - [PDF 재무 보고서](#pdf-report)
3. [사용 사례 2: Portfolio 분석 Workflow](#portfolio-analysis)
   - [Portfolio Analytics Excel](#portfolio-excel)
   - [투자위원회 Deck](#investment-deck)
4. [사용 사례 3: 자동 보고 Pipeline](#reporting-pipeline)

## 사전 요구 사항

이 notebook은 **Notebook 1: Skills 소개**를 완료했다고 가정합니다.

아직 완료하지 않았다면:
1. 먼저 Notebook 1의 설정 완료
2. 테스트 셀로 환경 검증
3. 파일 생성과 다운로드가 가능한지 확인

**필수:**
- Anthropic API key 설정 완료
- whl에서 설치한 SDK 버전 0.69.0
- Virtual environment 활성화

## 1. 설정 및 Data 로딩 {#setup}

dependency를 import하고 이 notebook 전체에서 사용할 재무 data를 불러오는 것으로 시작합니다.

In [ ]:
# 표준 import
import json
import os
import sys
from pathlib import Path

import pandas as pd

# import를 위해 상위 directory 추가
sys.path.insert(0, str(Path.cwd().parent))

# Anthropic SDK
from anthropic import Anthropic
from dotenv import load_dotenv

# 우리의 utility
from file_utils import (
    download_all_files,
    print_download_summary,
)

# 환경 로딩
load_dotenv(Path.cwd().parent / ".env")

# 설정
API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = "claude-sonnet-4-6"

if not API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. Please configure your .env file.")

# client 초기화
client = Anthropic(api_key=API_KEY)

# directory 설정
OUTPUT_DIR = Path.cwd().parent / "outputs" / "financial"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path.cwd().parent / "sample_data"

print("✓ Environment configured")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ Data directory: {DATA_DIR}")

### 재무 Data 로딩

회사의 재무 상태 중 서로 다른 측면을 나타내는 dataset 4개가 있습니다.

In [ ]:
# 재무제표 로딩
financial_statements = pd.read_csv(DATA_DIR / "financial_statements.csv")
print("📊 Financial Statements Overview:")
print(f"   Shape: {financial_statements.shape}")
print(f"   Categories: {len(financial_statements['Category'].unique())} financial metrics")
print(f"   Quarters: {list(financial_statements.columns[1:5])}")
print()

# sample data 표시
print("Sample data (first 5 rows):")
financial_statements.head()

In [ ]:
# portfolio 보유 자산 로딩
with open(DATA_DIR / "portfolio_holdings.json") as f:
    portfolio_data = json.load(f)

print("💼 Portfolio Overview:")
print(f"   Portfolio: {portfolio_data['portfolio_name']}")
print(f"   Total Value: ${portfolio_data['total_value']:,.2f}")
print(f"   Holdings: {len(portfolio_data['holdings'])} stocks")
print(f"   Cash Position: ${portfolio_data['cash_position']['amount']:,.2f}")
print(f"   Total Return: {portfolio_data['performance_metrics']['total_return_percent']:.1f}%")
print()

# 쉽게 조작하도록 holdings를 DataFrame으로 변환
portfolio_df = pd.DataFrame(portfolio_data["holdings"])
print("Top 5 holdings by value:")
portfolio_df.nlargest(5, "market_value")[["ticker", "name", "market_value", "unrealized_gain"]]

In [ ]:
# 분기별 metric 로딩
with open(DATA_DIR / "quarterly_metrics.json") as f:
    quarterly_metrics = json.load(f)

print("📈 Quarterly Metrics Overview:")
print(f"   Quarters available: {len(quarterly_metrics['quarters'])}")
print(f"   Metrics per quarter: {len(quarterly_metrics['quarters'][0])} KPIs")
print()

# 최근 분기 metric 표시
latest_quarter = quarterly_metrics["quarters"][-1]
print(f"Latest Quarter ({latest_quarter['quarter']}):")
for key, value in latest_quarter.items():
    if key != "quarter" and isinstance(value, int | float):
        if "revenue" in key.lower() or "cost" in key.lower():
            print(f"   {key.replace('_', ' ').title()}: ${value:,.0f}")
        elif "percent" in key.lower() or "margin" in key.lower() or "rate" in key.lower():
            print(f"   {key.replace('_', ' ').title()}: {value:.1f}%")
        else:
            print(f"   {key.replace('_', ' ').title()}: {value:,.0f}")

### Helper 함수

이 notebook의 helper 함수를 정의합니다.

In [ ]:
def create_skills_message(client, prompt, skills, prefix="", show_token_usage=True):
    """
    Skills로 message를 만드는 helper 함수입니다.

    Args:
        client: Anthropic client
        prompt: User prompt
        skills: skill dict 목록 [{"type": "anthropic", "skill_id": "xlsx", "version": "latest"}]
        prefix: 다운로드 파일의 prefix
        show_token_usage: token 사용 출력 여부

    Returns:
        (response, download_results) tuple
    """
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=4096,
        container={"skills": skills},
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[{"role": "user", "content": prompt}],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    if show_token_usage:
        print(
            f"\n📊 Token Usage: {response.usage.input_tokens} in, {response.usage.output_tokens} out"
        )

    # 파일 다운로드
    results = download_all_files(client, response, output_dir=str(OUTPUT_DIR), prefix=prefix)

    return response, results


def format_financial_value(value, is_currency=True, decimals=0):
    """표시용 재무 값의 형식을 지정합니다."""
    if is_currency:
        return f"${value:,.{decimals}f}"
    else:
        return f"{value:,.{decimals}f}"


print("✓ Helper functions defined")

## 2. 사용 사례 1: 재무 Dashboard 생성 {#financial-dashboard}

data를 불러오고 helper 함수를 정의했으니 첫 실용적 사용 사례인 종합적인 재무 dashboard 생성으로 들어가겠습니다. formula, 서식, chart를 자동으로 포함하는 multi-sheet Excel workbook 생성부터 시작합니다.

### 2.1 Excel 재무 Model {#excel-model}

다음이 포함된 재무 dashboard를 만듭니다.
- 전년 대비 비교가 있는 손익계산서
- 재무상태표 분석
- 현금흐름 추적
- 시각화가 있는 KPI dashboard

이는 일반적으로 수작업으로 여러 시간이 필요한 복잡한 Excel 생성 작업을 Claude Skills가 처리하는 방법을 보여 줍니다.

In [ ]:
# 재무 Dashboard Excel 생성
print("Creating financial dashboard Excel file...")
print("This creates a 2-sheet dashboard optimized for the Skills API.")
print("\n⏱️ Generation time: 1-2 minutes\n")

# 재무 data 준비
fs_data = financial_statements.to_dict("records")
quarters_2024 = ["Q1_2024", "Q2_2024", "Q3_2024", "Q4_2024"]

# 핵심 재무 metric 추출
revenue_by_quarter = {
    "Q1 2024": financial_statements[financial_statements["Category"] == "Revenue"][
        "Q1_2024"
    ].values[0],
    "Q2 2024": financial_statements[financial_statements["Category"] == "Revenue"][
        "Q2_2024"
    ].values[0],
    "Q3 2024": financial_statements[financial_statements["Category"] == "Revenue"][
        "Q3_2024"
    ].values[0],
    "Q4 2024": financial_statements[financial_statements["Category"] == "Revenue"][
        "Q4_2024"
    ].values[0],
}

financial_dashboard_prompt = f"""
Create a financial dashboard Excel workbook with 2 sheets:

Sheet 1 - "P&L Summary":
Create a Profit & Loss summary table for 2024 quarters with these rows:
- Revenue: {", ".join([f"Q{i + 1}: ${v / 1000000:.1f}M" for i, v in enumerate(revenue_by_quarter.values())])}
- Gross Profit: Use values from the data
- Operating Income: Use values from the data
- Net Income: Use values from the data
- Add a Total column with SUM formulas
- Add a row showing profit margins (Net Income / Revenue)
- Apply currency formatting and bold headers
- Add a simple bar chart showing quarterly revenue

Sheet 2 - "Key Metrics":
Create a metrics dashboard with:
- Total Revenue 2024: SUM of all quarters
- Average Quarterly Revenue: AVERAGE formula
- Q4 vs Q1 Growth: Percentage increase
- Best Quarter: MAX formula to identify
- Operating Margin Q4: Calculate from data
- Year-over-year growth vs 2023

Apply professional formatting with borders, bold headers, and currency formats.
"""

# Excel 재무 dashboard 생성
excel_response, excel_results = create_skills_message(
    client,
    financial_dashboard_prompt,
    [{"type": "anthropic", "skill_id": "xlsx", "version": "latest"}],
    prefix="financial_dashboard_",
)

print("\n" + "=" * 60)
print_download_summary(excel_results)

if len(excel_results) > 0 and excel_results[0]["success"]:
    print("\n✅ Financial dashboard Excel created successfully!")

### 💡 Excel 생성을 위한 모범 사례

테스트를 바탕으로 Skills로 Excel 파일을 만드는 최적의 접근 방식은 다음과 같습니다.

**권장 접근 방식:**
- **workbook당 sheet 2~3개**는 안정적으로 작동하고 빠르게 생성됩니다.
- **각 sheet를** 특정 목적에 집중하세요(예: P&L, metric, chart).
- **복잡성을 조금씩 추가**하세요. 간단하게 시작하고 개선하세요.

**복잡한 Dashboard:**
1. 복잡한 파일 하나 대신 **목적이 명확한 파일 여러 개 생성**
   - 예제: `financial_pnl.xlsx`, `balance_sheet.xlsx`, `kpi_dashboard.xlsx`
2. 파일을 순차적으로 만들고 개선하는 **pipeline 패턴 사용**
3. 필요하면 pandas나 openpyxl로 **programmatically 파일 결합**

**성능 팁:**
- 간단한 2-sheet dashboard: 약 1~2분
- PowerPoint와 PDF 생성: 복잡한 content에서 매우 안정적
- Token 사용: 구조화된 data(JSON/CSV)가 일반 글보다 효율적

### 2.2 경영진 PowerPoint {#executive-ppt}

재무 data가 Excel에 정리되었으니 핵심 insight를 요약하는 경영진 presentation을 만듭니다. Skills가 chart, 서식 있는 텍스트, 여러 slide로 전문적인 PowerPoint presentation을 생성하는 방법을 보여 줍니다. 이사회나 투자자 현황 보고에 적합합니다.

presentation에 포함되는 것:
- 2024년 Q4 실적 주요 내용
- 전년 대비 비교가 있는 재무 metric
- 시각화된 수익성 추세
- 핵심 요점과 전망

In [ ]:
print("Creating executive presentation from financial metrics...")
print("\n⏱️ Generation time: 1-2 minutes\n")

# presentation의 핵심 metric 계산
q4_2024_revenue = 14500000
q4_2023_revenue = 12300000
yoy_growth = (q4_2024_revenue - q4_2023_revenue) / q4_2023_revenue * 100

q4_2024_net_income = 1878750
q4_2023_net_income = 1209000
net_income_growth = (q4_2024_net_income - q4_2023_net_income) / q4_2023_net_income * 100

executive_ppt_prompt = f"""
Create a 4-slide executive presentation for Q4 2024 financial results:

Slide 1 - Title:
- Title: "Q4 2024 Financial Results"
- Subtitle: "Executive Summary - Acme Corporation"
- Date: January 2025

Slide 2 - Financial Highlights:
- Title: "Q4 2024 Performance Highlights"
- Create a two-column layout:
  Left side - Key Metrics:
  • Revenue: $14.5M (+{yoy_growth:.1f}% YoY)
  • Net Income: $1.88M (+{net_income_growth:.1f}% YoY)
  • Operating Margin: 17.9% (up 2.9pp)
  • Operating Cash Flow: $2.85M

  Right side - Column chart showing quarterly revenue:
  Q1 2024: $12.5M
  Q2 2024: $13.2M
  Q3 2024: $13.8M
  Q4 2024: $14.5M

Slide 3 - Profitability Trends:
- Title: "Margin Expansion & Profitability"
- Add a line chart showing net margin % by quarter:
  Q1 2024: 11.4%
  Q2 2024: 11.8%
  Q3 2024: 12.4%
  Q4 2024: 13.0%
- Add bullet points below:
  • Consistent margin expansion throughout 2024
  • Operating leverage driving profitability
  • Cost optimization initiatives delivering results

Slide 4 - Key Takeaways:
- Title: "Key Takeaways & Outlook"
- Bullet points:
  ✓ Record Q4 revenue of $14.5M
  ✓ 17.9% YoY revenue growth
  ✓ 55% increase in net income YoY
  ✓ Strong cash generation: $2.85M operating cash flow
  ✓ Well-positioned for continued growth in 2025

Use professional corporate design:
- Dark blue (#003366) for headers
- Clean, modern layout
- Data-driven visualizations
"""

# 경영진 presentation 생성
ppt_response, ppt_results = create_skills_message(
    client,
    executive_ppt_prompt,
    [{"type": "anthropic", "skill_id": "pptx", "version": "latest"}],
    prefix="executive_summary_",
)

print("\n" + "=" * 60)
print_download_summary(ppt_results)

if len(ppt_results) > 0 and ppt_results[0]["success"]:
    print("\n✅ Executive presentation created successfully!")

## 3. 사용 사례 2: Portfolio 분석 Workflow {#portfolio-analysis}

이제 회사 재무에서 투자 portfolio 분석으로 초점을 옮깁니다. 이 섹션에서는 앞서 불러온 portfolio data로 종합적인 portfolio analytics와 투자위원회 presentation을 만드는 방법을 보여 줍니다.

이 workflow가 보여 주는 것:
- Excel에서 상세 portfolio 성과 분석
- risk metric과 sector 배분 시각화
- 전문적인 투자위원회 presentation
- data 기반 rebalancing 권고

먼저 portfolio analytics Excel workbook을 만든 다음 발견 사항을 요약하는 투자위원회 presentation을 생성합니다.

### 먼저 종합적인 portfolio 분석 Excel workbook을 만듭니다.

투자위원회 presentation을 만들기 전에 portfolio data를 상세히 분석해야 합니다. 이 Excel workbook은 투자 권고의 기반이 됩니다.

In [ ]:
print("Creating portfolio analysis Excel workbook...")
print("This creates a focused 2-sheet portfolio analysis optimized for the Skills API.")
print("\n⏱️ Generation time: 1-2 minutes\n")

# 프롬프트용 portfolio data 준비
top_holdings = portfolio_df.nlargest(5, "market_value")
sector_allocation = portfolio_data["sector_allocation"]

portfolio_excel_prompt = f"""
Create a portfolio analysis Excel workbook with 2 sheets:

Sheet 1 - "Portfolio Overview":
Create a comprehensive holdings and performance table:

Section 1 - Holdings (top of sheet):
{portfolio_df[["ticker", "name", "shares", "current_price", "market_value", "unrealized_gain", "allocation_percent"]].head(10).to_string()}

Section 2 - Portfolio Summary:
- Total portfolio value: ${portfolio_data["total_value"]:,.2f}
- Total unrealized gain: ${portfolio_df["unrealized_gain"].sum():,.2f}
- Total Return: {portfolio_data["performance_metrics"]["total_return_percent"]:.1f}%
- YTD Return: {portfolio_data["performance_metrics"]["year_to_date_return"]:.1f}%
- Sharpe Ratio: {portfolio_data["performance_metrics"]["sharpe_ratio"]:.2f}
- Portfolio Beta: {portfolio_data["performance_metrics"]["beta"]:.2f}

Apply conditional formatting: green for gains, red for losses.
Add a bar chart showing top 5 holdings by value.

Sheet 2 - "Sector Analysis & Risk":
Create sector allocation and risk metrics:

Section 1 - Sector Allocation:
{json.dumps(sector_allocation, indent=2)}
Include a pie chart of sector allocation.

Section 2 - Key Risk Metrics:
- Portfolio Beta: {portfolio_data["performance_metrics"]["beta"]:.2f}
- Standard Deviation: {portfolio_data["performance_metrics"]["standard_deviation"]:.1f}%
- Value at Risk (95%): $62,500
- Maximum Drawdown: -12.3%
- Sharpe Ratio: {portfolio_data["performance_metrics"]["sharpe_ratio"]:.2f}

Section 3 - Rebalancing Recommendations:
- Reduce Technology from 20% to 18%
- Increase Healthcare from 8.7% to 10%
- Maintain current diversification

Apply professional formatting with clear sections and headers.
"""

# portfolio 분석 Excel 생성
portfolio_response, portfolio_results = create_skills_message(
    client,
    portfolio_excel_prompt,
    [{"type": "anthropic", "skill_id": "xlsx", "version": "latest"}],
    prefix="portfolio_analysis_",
)

print("\n" + "=" * 60)
print_download_summary(portfolio_results)

if len(portfolio_results) > 0 and portfolio_results[0]["success"]:
    print("\n✅ Portfolio analysis Excel created successfully!")

### 3.2 투자위원회 Presentation {#investment-deck}

상세 portfolio 분석을 완료했으니 투자위원회를 위한 전문적인 presentation을 만듭니다. Excel 분석의 핵심 insight를 의사결정자에 적합한 간결하고 시각적인 형식으로 정리합니다.

다룰 내용:
- 핵심 metric을 포함한 portfolio 성과 요약
- 자산 배분과 분산 분석
- risk metric과 위험 조정 수익률
- rebalancing을 위한 전략적 권고

In [ ]:
print("Creating investment committee presentation...")
print("\n⏱️ Generation time: 1-2 minutes\n")

investment_deck_prompt = f"""
Create a 5-slide investment committee presentation:

Slide 1 - Title:
- Title: "Portfolio Review - Q4 2024"
- Subtitle: "{portfolio_data["portfolio_name"]}"
- Date: January 2025
- Portfolio Value: ${portfolio_data["total_value"]:,.0f}

Slide 2 - Portfolio Overview:
- Title: "Portfolio Performance Summary"
- Two-column layout:

  Left - Key Metrics:
  • Total Value: ${portfolio_data["total_value"]:,.0f}
  • YTD Return: +{portfolio_data["performance_metrics"]["year_to_date_return"]:.1f}%
  • Total Return: ${portfolio_data["performance_metrics"]["total_return"]:,.0f}
  • Sharpe Ratio: {portfolio_data["performance_metrics"]["sharpe_ratio"]:.2f}

  Right - Bar chart of top 5 holdings by value:
  {", ".join([f"{h['ticker']}: ${h['market_value']:,.0f}" for h in top_holdings.to_dict("records")])}

Slide 3 - Sector Allocation:
- Title: "Asset Allocation & Diversification"
- Pie chart showing:
  Technology: {sector_allocation["Technology"]:.1f}%
  Financials: {sector_allocation["Financials"]:.1f}%
  Healthcare: {sector_allocation["Healthcare"]:.1f}%
  Consumer: {sector_allocation["Consumer Discretionary"] + sector_allocation["Consumer Staples"]:.1f}%
  Fixed Income: {sector_allocation["Bonds"]:.1f}%
  Cash: {sector_allocation["Cash"]:.1f}%

Slide 4 - Risk Analysis:
- Title: "Risk Metrics & Analysis"
- Content:
  Risk Indicators:
  • Portfolio Beta: {portfolio_data["performance_metrics"]["beta"]:.2f} (lower market risk)
  • Standard Deviation: {portfolio_data["performance_metrics"]["standard_deviation"]:.1f}%
  • Maximum Drawdown: -12.3%
  • Value at Risk (95%): $62,500

  Risk-Adjusted Performance:
  • Sharpe Ratio: {portfolio_data["performance_metrics"]["sharpe_ratio"]:.2f} (excellent)
  • Alpha Generation: +2.3% vs benchmark

Slide 5 - Recommendations:
- Title: "Strategic Recommendations"
- Bullet points:
  ✓ Maintain current allocation - well diversified
  ✓ Consider profit-taking in Technology (20% → 18%)
  ✓ Increase Healthcare allocation (8.7% → 10%)
  ✓ Monitor bond duration given rate environment
  ✓ Rebalance quarterly to maintain targets

Use professional investment presentation design.
"""

# 투자위원회 deck 생성
investment_response, investment_results = create_skills_message(
    client,
    investment_deck_prompt,
    [{"type": "anthropic", "skill_id": "pptx", "version": "latest"}],
    prefix="investment_committee_",
)

print("\n" + "=" * 60)
print_download_summary(investment_results)
print("\n✅ Investment committee presentation created successfully!")

## 4. 사용 사례 3: 자동 보고 Pipeline {#reporting-pipeline}

지금까지 특정 목적의 개별 문서를 만들었습니다. 이제 자동 workflow에서 여러 Skills를 연결하는 힘을 보여 줍니다. 이 pipeline 패턴은 같은 data source에서 여러 관련 문서를 생성해야 하는 production 시스템에 필수적입니다.

이 예제에서는 다음을 수행하는 완전한 보고 suite를 만듭니다.
1. Excel에서 계산과 chart로 **data 분석**
2. PowerPoint presentation에서 **insight 요약**
3. 정식 PDF 보고서에서 **과정 문서화**

이는 전통적으로 여러 도구와 수동 조율이 필요했던 종합적인 보고 해결책을 Skills가 함께 만드는 방법을 보여 줍니다.

**pipeline 접근 방식의 핵심 이점:**
- 모든 문서에서 일관된 data
- 전체 생성 시간 감소
- Token 사용 최적화
- 여러 보고서 유형으로 확장 가능

**⏱️ 예상 전체 시간:** 전체 pipeline 약 2~3분

In [ ]:
print("🔄 Starting Automated Reporting Pipeline")
print("=" * 60)
print("This will create a complete reporting suite:")
print("1. Excel analysis → 2. PowerPoint summary → 3. PDF documentation")
print("\n⏱️ Total pipeline time: 2-3 minutes\n")

# pipeline 전반의 token 사용 추적
pipeline_tokens = {"input": 0, "output": 0}

# 1단계: Excel 분석 생성
print("Step 1/3: Creating Excel analysis with quarterly metrics...")

excel_pipeline_prompt = f"""
Create a quarterly business metrics Excel file:

Sheet 1 - "Quarterly KPIs":
Create a table with these quarterly metrics for 2024:
{
    json.dumps(
        [
            {
                k: v
                for k, v in q.items()
                if k in ["quarter", "revenue", "gross_margin", "customer_count", "churn_rate"]
            }
            for q in quarterly_metrics["quarters"]
        ],
        indent=2,
    )
}

Add:
- Quarter-over-quarter growth calculations
- Average and total rows
- Conditional formatting for trends
- Line chart showing revenue trend
- Column chart showing customer count

Sheet 2 - "YoY Comparison":
Compare Q4 2024 vs Q4 2023 for all metrics.
Calculate percentage changes and highlight improvements.

Professional formatting with headers and borders.
"""

excel_response, excel_results = create_skills_message(
    client,
    excel_pipeline_prompt,
    [{"type": "anthropic", "skill_id": "xlsx", "version": "latest"}],
    prefix="pipeline_1_metrics_",
    show_token_usage=False,
)

pipeline_tokens["input"] += excel_response.usage.input_tokens
pipeline_tokens["output"] += excel_response.usage.output_tokens
print(
    f"✓ Excel created - Tokens: {excel_response.usage.input_tokens} in, {excel_response.usage.output_tokens} out"
)

# 2단계: PowerPoint 요약 생성
print("\nStep 2/3: Creating PowerPoint summary from metrics...")

ppt_pipeline_prompt = """
Create a 3-slide quarterly metrics summary presentation:

Slide 1:
- Title: "Q4 2024 Metrics Summary"
- Subtitle: "Automated Reporting Pipeline Demo"

Slide 2:
- Title: "Key Performance Indicators"
- Show Q4 2024 metrics:
  • Revenue: $3.2M (+15% QoQ)
  • Customers: 850 (+8.9% QoQ)
  • Gross Margin: 72%
  • Churn Rate: 2.8% (improved from 3.5%)
- Add a simple bar chart comparing Q3 vs Q4 revenue

Slide 3:
- Title: "Quarterly Trend Analysis"
- Line chart showing revenue growth Q1-Q4
- Key insight bullets:
  • Consistent QoQ growth
  • Customer acquisition accelerating
  • Churn reduction successful

Clean, data-focused design.
"""

ppt_response, ppt_results = create_skills_message(
    client,
    ppt_pipeline_prompt,
    [{"type": "anthropic", "skill_id": "pptx", "version": "latest"}],
    prefix="pipeline_2_summary_",
    show_token_usage=False,
)

pipeline_tokens["input"] += ppt_response.usage.input_tokens
pipeline_tokens["output"] += ppt_response.usage.output_tokens
print(
    f"✓ PowerPoint created - Tokens: {ppt_response.usage.input_tokens} in, {ppt_response.usage.output_tokens} out"
)

# 3단계: PDF 문서 생성
print("\nStep 3/3: Creating PDF documentation...")

pdf_pipeline_prompt = """
Create a PDF document summarizing the quarterly reporting pipeline:

AUTOMATED REPORTING PIPELINE
Q4 2024 Results Documentation

EXECUTIVE SUMMARY
This document summarizes the Q4 2024 business metrics generated through
our automated reporting pipeline.

KEY METRICS
- Revenue: $3.2M (15% QoQ growth)
- Customer Base: 850 active customers
- Gross Margin: 72%
- Churn Rate: 2.8% (improved from 3.5%)

PIPELINE COMPONENTS
1. Data Processing: Quarterly metrics analyzed in Excel
2. Visualization: Key insights presented in PowerPoint
3. Documentation: Formal report generated in PDF

AUTOMATION BENEFITS
• Reduced reporting time by 90%
• Consistent format and quality
• Eliminated manual errors
• Scalable to multiple reports

NEXT STEPS
- Expand pipeline to include predictive analytics
- Add automated email distribution
- Implement real-time data feeds

Generated: January 2025
Pipeline Version: 1.0

Format as a professional technical document.
"""

pdf_response, pdf_results = create_skills_message(
    client,
    pdf_pipeline_prompt,
    [{"type": "anthropic", "skill_id": "pdf", "version": "latest"}],
    prefix="pipeline_3_documentation_",
    show_token_usage=False,
)

pipeline_tokens["input"] += pdf_response.usage.input_tokens
pipeline_tokens["output"] += pdf_response.usage.output_tokens
print(
    f"✓ PDF created - Tokens: {pdf_response.usage.input_tokens} in, {pdf_response.usage.output_tokens} out"
)

# Pipeline 요약
print("\n" + "=" * 60)
print("🎯 PIPELINE COMPLETE!")
print("=" * 60)

print("\n📊 Pipeline Token Usage Summary:")
print(f"   Total Input Tokens: {pipeline_tokens['input']:,}")
print(f"   Total Output Tokens: {pipeline_tokens['output']:,}")
print(f"   Total Tokens: {pipeline_tokens['input'] + pipeline_tokens['output']:,}")
print(f"   Average per document: {(pipeline_tokens['input'] + pipeline_tokens['output']) // 3:,}")

print("\n📁 Generated Files:")
all_results = excel_results + ppt_results + pdf_results
for i, result in enumerate(all_results, 1):
    if result["success"]:
        print(f"   {i}. {os.path.basename(result['output_path'])} ({result['size'] / 1024:.1f} KB)")

print("\n✅ Automated reporting pipeline executed successfully!")
print("   All three documents created and linked in workflow.")

## 요약 및 다음 단계

### 달성한 것

이 notebook에서 다음 방법을 배웠습니다.

✅ **재무 Dashboard 생성**
- formula와 chart가 있는 multi-sheet Excel model 구축
- 경영진 PowerPoint presentation 생성
- 전문적인 PDF 보고서 생성

✅ **Portfolio 분석**
- portfolio analytics workbook 개발
- 투자위원회 presentation 생성
- risk metric과 rebalancing 도구 구현

✅ **자동 Pipeline**
- 여러 문서 형식 연결
- token 사용 최적화
- production에 사용할 수 있는 패턴 구축

### 핵심 요점

1. **Skills는 재무 문서 생성을 크게 단순화합니다** - 수작업으로 몇 시간 걸릴 일이 몇 분 걸립니다.
2. **Token 효율이 뛰어납니다** - Skills는 수동 지시보다 약 90% 적은 token을 사용합니다.
3. **품질이 전문적인 수준입니다** - 문서를 업무에서 즉시 사용할 수 있습니다.
4. **자동화가 간단합니다** - Pipeline 패턴은 복잡한 workflow를 가능하게 합니다.

### 학습 계속하기

📚 **다음: [Notebook 3 - Custom Skills 개발](03_skills_custom_development.ipynb)**
- 자신의 특화 재무 skill 구축
- 회사별 template 생성
- 고급 automation 구현

### 실험해 보세요

1. 자신의 metric을 포함하도록 **재무 dashboard 수정**
2. 다른 자산군으로 **custom portfolio 생성**
3. 구체적인 보고 요구를 위한 **pipeline 구축**
4. 생성 시간을 이해하도록 **복잡성 실험**
5. 서로 다른 문서 유형의 **token 사용 추적**

### Resource

- [Claude API 문서](https://docs.anthropic.com/en/api/messages)
- [Skills 문서](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)
- [모범 사례](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [Files API Reference](https://docs.claude.com/en/api/files-content)